# Repo and secured financing

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Prerequisites:** `02_pricing/pricing_fundamentals.ipynb`. **`repo`** prices collateralized borrowing with **haircuts** and **repo rate** inputs.


## Concept

- **Collateral + haircut** govern lent securities vs cash.
- **Term vs overnight** changes accrual horizon and balance-sheet usage.
- **Implied financing** metrics summarize the carry embedded in the box.


In [1]:
import json

import sys
sys.path.insert(0, "../..")

from _shared import DEMO_AS_OF, print_metrics
from finstack_quant.valuations import ValuationResult
from finstack_quant.valuations.instruments import price_instrument_with_metrics, validate_instrument_json
from finstack_quant.core.market_data import DiscountCurve, MarketContext

print("Imports OK (repo).")


Imports OK (repo).


## Instrument JSON

**Collateral vs cash + haircut:** `quantity` × collateral mark (`market_value_id`) should cover **`cash_amount` × (1 + haircut)`** (cash lent plus the haircut buffer). Here `ceil(10.2e6 / 98.5) ≈ 103_554` units at **98.5** per unit.


In [2]:
AS_OF = DEMO_AS_OF
AS_OF_STR = AS_OF.isoformat()

repo = json.dumps(
    {
        "type": "repo",
        "spec": {
            "id": "REPO-GC-7D",
            "repo_type": "Term",
            "start_date": AS_OF_STR,
            "maturity": "2025-01-22",
            "day_count": "Act360",
            "bdc": "following",
            "calendar_id": "usny",
            "repo_rate": "0.0525",
            "haircut": 0.02,
            "cash_amount": {"amount": "10000000", "currency": "USD"},
            "collateral": {
                "collateral_type": "General",
                "instrument_id": "UST-10Y",
                "market_value_id": "UST_10Y_PRICE",
                "quantity": 103554.0,
            },
            "discount_curve_id": "USD-OIS",
            "triparty": False,
            "attributes": {},
            "pricing_overrides": {},
        },
    }
)

validate_instrument_json(repo)
print("repo JSON OK")


repo JSON OK


## MarketContext


In [3]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.02, 0.999), (1.0, 0.97)],
    day_count="act_365f",
)
mc = MarketContext().insert(ois)
mc.insert_price("UST_10Y_PRICE", 98.5, "USD")
market_json = mc.to_json()
print("collateral price id UST_10Y_PRICE inserted")

collateral price id UST_10Y_PRICE inserted


## Pricing


In [4]:
out = price_instrument_with_metrics(repo, market_json, AS_OF_STR, model="discounting")
print(ValuationResult.from_json(out))


ValuationResult(id="REPO-GC-7D", price=603.0547, currency=USD, metrics=0)


## Metrics


In [5]:
metrics = ["implied_financing_rate", "funding_cost", "collateral_haircut01"]
out = price_instrument_with_metrics(
    repo,
    market_json,
    AS_OF_STR,
    model="discounting",
    metrics=metrics,
)
print_metrics(ValuationResult.from_json(out), metrics)


  funding_cost: 0.0
  collateral_haircut01: 0.0


## Takeaways

- **Haircut** and **collateral marks** (`market_value_id`) drive security-for-cash economics.
- **Implied financing rate** links observed market levels to the repo structure.
- Overnight vs term is mostly a **schedule + rate** choice in JSON.
